In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydicom import dcmread

warnings.filterwarnings('ignore')
plt.rcParams['image.interpolation'] = 'nearest'

STUDY_ROOT = Path('data/train_series')
CSV_CANDIDATES = [
    Path('data/train_series.csv'),
]

LABEL_COLUMNS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

def load_series_metadata():
    for csv_path in CSV_CANDIDATES:
        if csv_path.exists():
            df = pd.read_csv(csv_path, dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
            df = df.drop_duplicates('SeriesInstanceUID').set_index('SeriesInstanceUID')
            return df
    return None

METADATA = load_series_metadata()

print('STUDY_ROOT:', STUDY_ROOT)
print('STUDY_ROOT exists:', STUDY_ROOT.exists())
print('metadata loaded:', METADATA is not None)
if METADATA is not None:
    print(METADATA.head())


In [ ]:
df = pd.read_csv(
            "data/train.csv",
            usecols=["StudyInstanceUID"] + LABEL_COLUMNS,
            dtype={"StudyInstanceUID": str},
        )

pseudo_df = pd.read_csv(
            "data/llm_labels_full.csv",
            usecols=["StudyInstanceUID"] + LABEL_COLUMNS,
            dtype={"StudyInstanceUID": str},
        )

labels_raw = df[LABEL_COLUMNS]

# Приводим к numeric. Пропуссы останутся NaN.
labels = labels_raw.apply(lambda col: pd.to_numeric(col, errors="coerce"))

# "Присутствует label" = не NaN, независимо от того, 0 или 1.
label_present = labels.notna()

# Количество заполненных label-ов в строке.
num_present_labels = label_present.sum(axis=1)

# Количество положительных label-ов в строке.
# Если нужно именно "есть хотя бы один положительный признак".
num_positive_labels = (labels > 0.5).sum(axis=1)

n_labels = len(LABEL_COLUMNS)

total_rows = len(df)

rows_no_labels = int((num_present_labels == 0).sum())
rows_at_least_one_label = int((num_present_labels >= 1).sum())
rows_fully_labeled = int((num_present_labels == n_labels).sum())
rows_partially_labeled = int(((num_present_labels >= 1) & (num_present_labels < n_labels)).sum())

rows_with_positive = int((num_positive_labels >= 1).sum())
rows_fully_labeled_with_positive = int(
    ((num_present_labels == n_labels) & (num_positive_labels >= 1)).sum()
)

print("\n=== Label coverage ===")
print("Rows with 0 present labels:", rows_no_labels)
print("Rows with >=1 present label:", rows_at_least_one_label)
print("Rows partially labeled:", rows_partially_labeled)
print("Rows fully labeled:", rows_fully_labeled)

if total_rows > 0:
    print("\n=== Percentages ===")
    print(f"0 labels: {rows_no_labels / total_rows:.2%}")
    print(f">=1 label: {rows_at_least_one_label / total_rows:.2%}")
    print(f"partial: {rows_partially_labeled / total_rows:.2%}")
    print(f"fully labeled: {rows_fully_labeled / total_rows:.2%}")

print("\n=== Positive labels ===")
print("Rows with at least one positive label:", rows_with_positive)
print("Rows fully labeled and with at least one positive label:", rows_fully_labeled_with_positive)

print("\n=== Distribution: number of present labels per row ===")
print(num_present_labels.value_counts().sort_index())

print("\n=== Present count per label column ===")
print(label_present.sum())

print("\n=== Positive count per label column ===")
print(labels.sum(axis=0, skipna=True))

print("\n=== Raw dtypes ===")
print(labels_raw.dtypes)

print("\n=== Sample raw values for first 3 label columns ===")
for col in LABEL_COLUMNS:
    print(col)
    print(df[col].value_counts(dropna=False).head(10))
    print("-" * 40)

print("\n=== Example partially labeled rows ===")
partial_mask = (num_present_labels >= 1) & (num_present_labels < n_labels)
partial_df = df[partial_mask]
print("Partially labeled rows:", len(partial_df))
print(partial_df[["StudyInstanceUID"] + LABEL_COLUMNS].head(10))

In [ ]:
def split_data(df_full, df_pseudo, label_columns):
    """
    df_full: Полный датасет с частичной разметкой (data/train.csv)
    df_pseudo: Датасет с псевдолейблами (или полный набор размеченных + псевдо)
    label_columns: Список колонок с лейблами
    """
    
    # --- ШАГ 1: Формируем Val DF из полного датасета ---
    
    # Находим строки, где есть хотя бы одна заполненная метка
    # Используем ту же логику, что и в вашем коде: notna().any(axis=1)
    mask_labeled = df_full[label_columns].notna().any(axis=1)
    
    val_df = df_full[mask_labeled].copy()
    
    print(f"Размер Val DF (размеченные из полного набора): {len(val_df)}")
    
    # Получаем список UID, которые ушли в валидацию
    val_uids = set(val_df['StudyInstanceUID'].unique())
    
    # --- ШАГ 2: Формируем Train DF из псевдолейблов ---
    
    # Исключаем из псевдолейблов те исследования, которые уже есть в валидации
    # Это гарантирует, что одно и то же исследование не будет и в train, и в val
    train_df = df_pseudo[~df_pseudo['StudyInstanceUID'].isin(val_uids)].copy()
    
    print(f"Размер Train DF (псевдолейблы минус val): {len(train_df)}")
    
    return train_df, val_df

# Пример использования:
train_df_final, val_df_final = split_data(df, pseudo_df, LABEL_COLUMNS)

val_df_final.head()

In [ ]:
def list_studies(root=STUDY_ROOT):
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])

def list_series(study_uid, root=STUDY_ROOT):
    study_dir = root / study_uid
    if not study_dir.exists():
        return []
    return sorted([p.name for p in study_dir.iterdir() if p.is_dir()])

def _sort_key(ds):
    try:
        return (0, int(ds.InstanceNumber))
    except Exception:
        pass
    try:
        if hasattr(ds, 'ImagePositionPatient') and hasattr(ds, 'ImageOrientationPatient'):
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float).reshape(2, 3)
            normal = np.cross(iop[0], iop[1])
            if np.linalg.norm(normal) > 1e-8:
                return (1, float(np.dot(normal, ipp)))
    except Exception:
        pass
    return (2, 0)

def load_series(study_uid, series_uid, root=STUDY_ROOT):
    series_dir = root / study_uid / series_uid
    if not series_dir.exists():
        return []
    files = sorted(series_dir.glob('*.dcm'))
    datasets = []
    for f in files:
        try:
            datasets.append(dcmread(f))
        except Exception as e:
            print(f'Ошибка чтения {f}: {e}')
    datasets.sort(key=_sort_key)
    return datasets

def get_pixel_array(ds):
    try:
        arr = np.asarray(ds.pixel_array, dtype=np.float32)
    except Exception as e:
        print('Не удалось получить pixel_array:', e)
        return None
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2') == 'MONOCHROME1':
        arr = arr.max() - arr
    return arr

def auto_clip(arr):
    if arr is None:
        return None
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return arr
    return np.clip(arr, p1, p99)

def series_meta_text(series_uid):
    if METADATA is not None and series_uid in METADATA.index:
        row = METADATA.loc[series_uid]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        plane = row.get('Anatomical_Plane', '')
        fluid = row.get('Fluid_Sensitive', '')
        fat = row.get('Fat_Suppression', '')
        return f'plane={plane}, fluid={fluid}, fat={fat}'
    return ''

STUDIES = list_studies()
print('Студий:', len(STUDIES))
for study_uid in STUDIES[:5]:
    series = list_series(study_uid)
    print(study_uid, '->', len(series), 'series')


In [ ]:
def show_series(study_uid, series_uid, n_images=16, figsize=(12, 12)):
    ds_list = load_series(study_uid, series_uid)
    if not ds_list:
        print('Серия пуста или не найдена.')
        return

    n = min(n_images, len(ds_list))
    idx = np.linspace(0, len(ds_list) - 1, n, dtype=int)
    cols = 4
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1:
        axes = axes.tolist()
    elif cols == 1:
        axes = [axes[i, 0] for i in range(rows)]
    else:
        axes = axes.ravel().tolist()

    for ax, i in zip(axes, idx):
        i = int(i)
        arr = auto_clip(get_pixel_array(ds_list[i]))
        if arr is None:
            ax.axis('off')
            continue
        ax.imshow(arr, cmap='gray', origin='upper')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'slice {i + 1}/{len(ds_list)}', fontsize=8)

    for ax in axes[len(idx):]:
        ax.axis('off')

    title = f'Study: {study_uid}\nSeries: {series_uid}\nSlices: {len(ds_list)}'
    meta = series_meta_text(series_uid)
    if meta:
        title += '\n' + meta

    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()

# Пример: первая серия первой студии
if STUDIES:
    study_uid = STUDIES[1]
    series_uid = list_series(study_uid)[1]
    show_series(study_uid, series_uid)
else:
    print('Данные не найдены. Проверь STUDY_ROOT.')
